**Процесс** — это программа, которая запущена в оперативной памяти компьютера. Другими словами, процесс — это **набор инструкций**, которые выполняются последовательно. Каждый процесс, который запущен в операционной системе, имеет свои характеристики. Одна из главных характеристик — это **идентификатор процесса или PID**. Кроме того, каждый процесс занимает некий объем оперативной памяти, которую он запрашивает у системы. Система возвращает запрошенный объём памяти процессу и аллоцирует её. Также у процесса есть стек — он используется для вызова функций и создания локальных переменных у этих функций. И, наконец, у каждого процесса есть список открытых файлов, стандартный ввод и стандартный вывод.

In [1]:
# простой Python процесс
# ps aux | grep PID
import time
import os

pid = os.getpid()

while True:
    print(pid, time.time())
    time.sleep(2)

61490 1669901820.82495
61490 1669901822.827113
61490 1669901824.8299336


KeyboardInterrupt: 

Процесс в операционной системе создается при помощи системного вызова fork. Давайте рассмотрим программу, которая создает дочерний процесс при помощи системного вызова fork

In [3]:
# Создание процесса на Python
import time
import os

pid = os.fork()

if pid == 0:
    # дочерний процесс
    while True:
        print("child:", os.getpid())
        time.sleep(5)
else:
    # родительский процесс
    print("parent:", os.getpid())
    os.wait()

child: 61537
parent: 61490
child: 61537


KeyboardInterrupt: 

Системный вызов fork создает точную копию родительского процесса. Это означает, что вся память, все файловые дескрипторы и все ресурсы, которые были доступны в родительском процессе, будут целиком и полностью скопированы в дочерний процесс. С того момента, как системный вызов fork отработал, у нас имеется два одинаковых процесса в операционной системе. Единственное отличие заключается в том, что системный вызов fork в родительский процесс вернет PID дочернего процесса, а в дочернем процессе переменная pid будет равна нулю.

In [1]:
# Память родительского и дочернего процесса
import os
import time

foo = "bar"

pid = os.fork()

if pid == 0:
# дочерний процесс
    foo = "baz"
    print("child:", foo)
    time.sleep(5)
else:
    # родительский процесс
    print("parent:", foo)
    os.wait()

child: baz
parent: bar


: 

: 

После того, как отработал системный вызов fork, как уже было сказано, вся память
целиком и полностью будет скопирована из родительского процесса в дочерний. Значит,
переменная foo будет доступна в дочернем процессе со значением "bar". Но если мы
изменим значение foo в дочернем процессе, это никак не повлияет на переменную foo,
которая была объявлена в родительском процессе

Все эти примеры носят обучающий характер, и обычно код с использованием системных вызовов fork немного сложнее. fork может вернуть ошибку, которую нужно проверять, поэтому обычно в Python-е для создания процессов используют модуль multiprocessing. Для того, чтобы запустить процесс таким способом, необходимо импортировать класс Process из модуля multiprocessing, создать объект класса Process, передать ему в конструктор функцию, которую мы хотим исполнить в отдельном дочернем процессе и аргументы этой функции. Процесс будет создан тогда, когда мы вызовем метод start нашего объекта. Внутри метода start будет вызван системный вызов fork и исполнена наша функция f в отдельном процессе. Очень важно ожидать завершения всех созданных дочерних процессов. Для этого можно воспользоваться удобной функцией join

In [1]:
# Создание процесса, модуль multiprocessing
from multiprocessing import Process

def f(name):
    print("hello", name)
    
p = Process(target=f, args=("Bob",))
p.start()
p.join()

hello Bob


### Потоки

Мы обсудим создание потоков при помощи модуля threading и использование класса ThreadPoolExecutor. С прикладной точки зрения поток целиком и полностью напоминает процесс. Он имеет свою последовательность инструкций для исполнения, у каждого потока есть свой собственный стек, но все потоки выполняются в рамках одного процесса. Этим они отличаются от процессов. Если, когда мы говорили о процессах, у каждого процесса были свои ресурсы и память, то все созданные потоки разделяют память процесса и все его ресурсы. Управлением и выполнением потоков занимается операционная система. Но в Python есть свои ограничения для потоков — их мы обсудим отдельно.

In [2]:
# Создание потока
from threading import Thread

def f(name):
    print("hello", name)
    
th = Thread(target=f, args=("Bob",))
th.start()
th.join()

hello Bob


Если вы запустите несколько потоков для решения своей задачи, то вам рано или поздно придётся обмениваться данными между потоками. Часто для синхронизации потоков используют блокировки. Но любые блокировки замедляют выполнение программы. Лучше не использовать блокировки и отдавать предпочтение обмену данными через очереди.

In [11]:
# Очереди, модуль queue
import time
from queue import Queue
from threading import Thread

def worker(q, n):
    while True:
        item = q.get()
        if item is None:
            break
        time.sleep(0.001)
        print("process data:", n, item)

q = Queue(5)
th1 = Thread(target=worker, args=(q, 1))
th2 = Thread(target=worker, args=(q, 2))
th1.start(); th2.start()

for i in range(50):
    q.put(i)
    
q.put(None); q.put(None)
th1.join(); th2.join()

process data: 2 0
process data: 1 1
process data: 2 2
process data: 1 3
process data: 2 4
process data: 1 5
process data: 2 6
process data: 1 7
process data: 2 8
process data: 1 9
process data: 2 10
process data: 1 11
process data: 2 12
process data: 1 13
process data: 2 14
process data: 1 15
process data: 2 16
process data: 1 17
process data: 2 18
process data: 1 19
process data: 2 20
process data: 1 21
process data: 2 22
process data: 1 23
process data: 2 24
process data: 1 25
process data: 2 26
process data: 1 27
process data: 2 28
process data: 1 29
process data: 2 30
process data: 1 31
process data: 2 32
process data: 1 33
process data: 2 34
process data: 1 35
process data: 2 36
process data: 1 37
process data: 2 38
process data: 1 39
process data: 2 40
process data: 1 41
process data: 2 42
process data: 1 43
process data: 2 44
process data: 1 45
process data: 2 46
process data: 1 47
process data: 2 48
process data: 1 49


Большое внимание нужно уделить правильному завершению потока. Ресурсами процесса, то есть выделенной памятью или открытым файлом владеет сам процесс. Но процесс ничего не знает о том, что делает с этими ресурсами поток. И если поток завершить аварийно, то файл может остаться незакрытым, блокировка — невысвобожденной, что теоретически может привести к непредвиденным последствиям. Поэтому в Python не существует функции аварийного завершения потока. Очень важно делать это правильно в функции самого потока

### Блокировки

Пять философов сидят за круглым столом с пятью тарелками спагетти (разновидность макарон) и пятью вилками, как показано на схеме.

В любой момент времени философ должен либо есть, либо думать.

Кроме того, философ должен взять две вилки рядом с ним (то есть левую и правую вилки), прежде чем он сможет съесть спагетти. Проблема тупика возникает, когда все пять философов одновременно поднимают свои правильные вилки.

Поскольку у каждого из философов есть одна вилка, они все будут ждать, пока другие положат свою вилку. В результате никто из них не сможет съесть спагетти.

![](https://coderlessons.com/wp-content/uploads/images/gur/b724495d7bdaf046a6e7ad0c43ddc6c8.png)

Потоки стремятся к ресурсам, с которыми должны работать. И когда к одному и тому же ресурсу обращается несколько потоков, возникает конфликт. Как его предотвратить?

Потоки нельзя в любой момент напрямую остановить или завершить: на то они и потоки. Но можно на их пути поставить дамбу — блокировку. Она пропустит только один поток, а остальные временно удержит. Так вы исключите конфликт.

Когда поток A выполняет операцию с общими ресурсами, а поток Б не может вмешаться в нее до завершения — говорят, что такая операция атомарна. Залогом потокобезопасности как раз выступает атомарность — непрерывность, неделимость операции.

In [44]:
import threading

protected_resource = 0
unprotected_resource = 0

NUM = 50000
mutex = threading.Lock()

# Потокобезопасный инкремент
def safe_plus():
    global protected_resource
    for i in range(NUM):
        # Ставим блокировку
        mutex.acquire()
        protected_resource += 1
        mutex.release()

# Потокобезопасный декремент
def safe_minus():
    global protected_resource
    for i in range(NUM):
        mutex.acquire()
        protected_resource -= 1
        mutex.release()

# То же, но без блокировки
def risky_plus():
    global unprotected_resource
    for i in range(NUM):
        unprotected_resource += 1

def risky_minus():
    global unprotected_resource
    for i in range(NUM):
        unprotected_resource -= 1

In [158]:
thread1 = threading.Thread(target = safe_plus)
thread2 = threading.Thread(target = safe_minus)
thread3 = threading.Thread(target = risky_plus)
thread4 = threading.Thread(target = risky_minus)
thread1.start()
thread2.start()
thread3.start()
thread4.start()
thread1.join()
thread2.join()
thread3.join()
thread4.join()
print ("Результат при работе с блокировкой %s" % protected_resource)
print ("Результат без блокировки %s" % unprotected_resource)

Результат при работе с блокировкой 0
Результат без блокировки 451564


In [162]:
import threading, time
 
resource = {'A': threading.Lock(), 'B': threading.Lock()}
def proc(n, rs):
    for r in rs:
        print("Процесс %s запрашивает ресурс %s" % (n, r))
        resource[r].acquire()
        print("Процесс %s получил ресурс %s" % (n, r))
        time.sleep(1)
    print("Процесс %s выполняется" % n)
    for r in rs:
        resource[r].release()
    print("Процесс %s закончил выполнение" % n)

p1 = threading.Thread(target=proc, name="t1", args=["1", "AB"])
p2 = threading.Thread(target=proc, name="t2", args=["2", "BA"])
p1.start()
p2.start()
p1.join()
p2.join()

Процесс 1 запрашивает ресурс A
Процесс 1 получил ресурс A
Процесс 2 запрашивает ресурс B
Процесс 2 получил ресурс B
Процесс 1 запрашивает ресурс BПроцесс 2 запрашивает ресурс A



KeyboardInterrupt: 

### GIL

Поговорим о том, что такое глобальная блокировка интерпретатора, или, как её сокращенно называют, GIL. GIL очень тесно связан с выполнением потоков. Многие разработчики знают о GIL лишь то, что это какая-то штука, которая не позволяет одновременно двум потокам выполняться на одном ядре процессора, даже если этих ядер у процессора несколько. Тем не менее, GIL в первую очередь предназначен для защиты памяти интерпретатора от разрушений и делает все операции с памятью атомарными

In [166]:
# cpu bound programm
from threading import Thread
import time

def count(n):
    while n > 0:
        n -= 1
    
# series run
t0 = time.time()
count(100_000_000)
count(100_000_000)
print(f"Синхронно - {time.time() - t0}")

# parallel run
t0 = time.time()
th1 = Thread(target=count, args=(100_000_000,))
th2 = Thread(target=count, args=(100_000_000,))
th1.start(); th2.start()
th1.join(); th2.join()
print(f"Параллельно - {time.time() - t0}")

Синхронно - 5.41705584526062
Параллельно - 10.892079830169678


![](https://uwpce-pythoncert.github.io/SystemDevelopment/_images/gil.png)

In [45]:
import urllib.request
import time
from concurrent.futures import ThreadPoolExecutor

urls = [
  'http://www.python.org',
  'https://docs.python.org/3/',
  'https://docs.python.org/3/whatsnew/3.7.html',
  'https://docs.python.org/3/tutorial/index.html',
  'https://docs.python.org/3/library/index.html',
  'https://docs.python.org/3/reference/index.html',
  'https://docs.python.org/3/using/index.html',
  'https://docs.python.org/3/howto/index.html',
  'https://docs.python.org/3/installing/index.html',
  'https://docs.python.org/3/distributing/index.html',
  'https://docs.python.org/3/extending/index.html',
  'https://docs.python.org/3/c-api/index.html',
  'https://docs.python.org/3/faq/index.html'
  ]

def open_url(url):
    print(f'Открываю {url}')
    urllib.request.urlopen(url)

t0 = time.time()
with ThreadPoolExecutor(14) as executor:
    results = executor.map(open_url, urls)
print(f"Параллельно - {time.time() - t0}")

Открываю http://www.python.org
Открываю https://docs.python.org/3/
Открываю https://docs.python.org/3/whatsnew/3.7.html
Открываю https://docs.python.org/3/tutorial/index.html
Открываю https://docs.python.org/3/library/index.html
Открываю https://docs.python.org/3/reference/index.html
Открываю https://docs.python.org/3/using/index.html
Открываю https://docs.python.org/3/howto/index.html
Открываю https://docs.python.org/3/installing/index.html
Открываю https://docs.python.org/3/distributing/index.html
Открываю https://docs.python.org/3/extending/index.html
Открываю https://docs.python.org/3/c-api/index.html
Открываю https://docs.python.org/3/faq/index.html
Параллельно - 0.5595159530639648


### Генераторы

Генератор это подвид итерируемых объектов, как список или кортеж. Он генерирует для нас последовательность значений, которую мы можем перебрать. Эту последовательность можно использовать для итерации в цикле for, но нельзя проиндексировать (т. е., перебрать ее можно только один раз).

In [5]:
def counter():
    i=1
    while(i<=10):
        yield i
        i+=1

In [6]:
for i in counter():
    print(i)

1
2
3
4
5
6
7
8
9
10


In [7]:
def my_gen(x):
    while( x> 0):
        if x%2==0:
            yield 'Even'
        else:
            yield 'Odd'
        x-=1
        
for i in my_gen(7):
      print(i)

Odd
Even
Odd
Even
Odd
Even
Odd


In [8]:
gen = my_gen(7)

In [18]:
next(gen)

StopIteration: 

#### Генератор vs. функция

- Генератор использует yield для отправления значения пользователю, а у функции для этого есть return;
- При использовании генератора вызовов yield может быть больше чем один;
- Вызов yield останавливает исполнение и возвращает итератор, а return всегда выполняется последним;
- Вызов метода next() приводит к выполнению функции генератора;
- Локальные переменные и состояния сохраняются между последовательными вызовами метода next();
- Каждый дополнительный вызов next() вызывает исключение StopIteration, если нет следующих элементов для обработки.

#### Когда использовать генератор?

- Генераторы помогают обрабатывать большие объемы данных. Они позволяют производить так называемые ленивые вычисления. Подобным образом происходит потоковая обработка.
- Генераторы можно устанавливать друг за другом и использовать их как Unix-каналы.
- Генераторы позволяют настроить одновременное исполнение
- Они часто используются для чтения крупных файлов. Это делает код чище и компактнее, разделяя процесс на более мелкие сущности.
- Генераторы особенно полезны для веб-скрапинга и увеличения эффективности поиска. Они позволяют получить одну страницу, выполнить какую-то операцию и двигаться к следующей. Этот подход куда эффективнее чем получение всех страниц сразу и использование отдельного цикла для их обработки.

### Сопрограммы/Корутины

Мы в бесконечном цикле вызываем такую конструкцию line присвоить yield (в отличие от генераторов в корутинах мы переменной присваиваем результат работы конструкции yield). В данном случае функция заморозит своё состояние и будет ожидать ввода данных при помощи метода send. Итак, если мы вызовем функцию grep, будет создана корутина. Для того чтобы запустить нашу корутину, необходимо вызвать метод next. После того, как метод next будет вызван, запустится код нашей функции, выведется строчка start grep, запустится бесконечный цикл, код дойдет до инструкции yield, и здесь управление вернется в основной поток. После этого в основном потоке мы отправляем данные нашей корутине, и код функции выполняется дальше:

In [19]:
# Сопрограммы (корутины)
def grep(pattern):
    print("start grep")
    while True:
        line = yield
        if pattern in line:
            print(line)

In [20]:
g = grep("python")
next(g) # g.send(None)
g.send("golang is better?")
g.send("python is simple!")

start grep
python is simple!


In [21]:
# Сопрограммы, вызов метода close()
def grep(pattern):
    print("start grep")
    try:
        while True:
            line = yield
            if pattern in line:
                print(line)
    except GeneratorExit:
        print("stop grep")

In [22]:
g = grep("python")
next(g) # g.send(None)
g.send("golang is better?")
g.send("python is simple!")
g.close()

start grep
python is simple!
stop grep


Для того чтобы было удобно вызывать из одних корутин другие, в Python разработали стандарт PEP 0380 и в Python 3 его реализовали. В новом стандарте появилась инструкция yield from. При помощи неё можно выполнить делегирование вызова одной корутины в другой. Наш пример можно переписать следующим образом мы используем инструкцию yield from и указываем объект в виде другой корутины

In [24]:
# Сопрограммы, yield from PEP 0380
def grep(pattern):
    print("start grep")
    while True:
        line = yield
        if pattern in line:
            print(line)
    
def grep_python_coroutine():
    g = grep("python")
    yield from g

In [25]:
g = grep_python_coroutine()
g.send(None)
g.send("python wow!")

start grep
python wow!


### Asyncio

asyncio — это библиотека, которая стала частью Python 3. Она отвечает за неблокирующий ввод/вывод, на этом фреймворке можно написать сервис, который работает с десятками тысяч соединений одновременно. В основе работы этого фреймворка лежат генераторы и корутины,

#### Разница в понятиях параллелизма, concurrency, поточности и асинхронности

Параллелизм — это выполнение нескольких операций за раз. Многопроцессорность — один из примеров. Отлично подходит для задач, нагружающих CPU.

Concurrency — более широкое понятие, которое описывает несколько задач, выполняющихся с перекрытием друг друга.

Поточность — поток — это отдельный поток выполнения. Один процесс может содержать несколько потоков, где каждый будет работать независимо. Отлично подходит для IO-операций.

Асинхронность — однопоточный, однопроцессорный дизайн, использующий многозадачность. Другими словами, асинхронность создает впечатление параллелизма, используя один поток в одном процессе.

In [27]:
import asyncio

async def hello_world():
    while True:
        print("Hello World!")
        await asyncio.sleep(1.0)
        
loop = asyncio.get_event_loop()
loop.run_until_complete(hello_world())
loop.close()

RuntimeError: This event loop is already running

Есть три варианта awaitable объектов:

- Другая сопрограмма, а именно объект native coroutine. Этот напоминает, и видимо реализовано аналогично случаю, когда в генераторе с помощью yield from вызывается другой генератор.
- Сопрограмма на основе генератора, созданная с помощью декоратора types.coroutine(). Это вариант обеспечения совместимости с наработками, где сопрограммы реализованы на основе генераторов.
- Специальный объект, у которого реализован магический метод __await__, возвращающий итератор. С помощью этого итератора реализуется возврат результата выполнения сопрограммы.

#### Задачи

Задачи используются для планирования параллельного выполнения сопрограмм.

При передаче сопрограммы в цикл событий для обработки можно получить объект Task, который предоставляет способ управления поведением сопрограммы извне цикла событий.

In [28]:
import asyncio


async def async_func():
    print('Запуск ...')
    await asyncio.sleep(1)
    print('... Готово!')


async def main():
    task = asyncio.create_task (async_func())
    await task

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

#### Циклы событий

Этот механизм выполняет сопрограммы до тех пор, пока те не завершатся. Это можно представить как цикл while True, который отслеживает сопрограммы, узнавая, когда те находятся в режиме ожидания, чтобы в этот момент выполнить что-нибудь другое.

Он может разбудить спящую сопрограмму в тот момент, когда она ожидает своего времени, чтобы выполниться. В одно время может выполняться лишь один цикл событий в Python.

In [29]:
### asyncio.Task, запуск нескольких корутин
import asyncio

async def sleep_task(num):
    for i in range(5):
        print(f"process task: {num} iter: {i}")
        await asyncio.sleep(1)
    return num

# ensure_future or create_task
loop = asyncio.get_event_loop()
task_list = [loop.create_task(sleep_task(i)) for i in range(2)]
loop.run_until_complete(asyncio.wait(task_list))
loop.close()

/usr/lib/python3.8/tracemalloc.py:137: RuntimeWarning: coroutine 'hello_world' was never awaited
  class Frame:
<ipython-input-29-5f11a0b45d6e>:22: RuntimeWarning: coroutine 'main' was never awaited
  pass


In [35]:
def download_file(url):
    for i in range(3):
        print(f"Скачиваю часть {i}")
        yield


image_downloader = download_file('https://www.some-images.com/image1.jpg')
music_downloader = download_file('https://www.music-site.com/artist/album/song5.mp3')

coroutines = [music_downloader, image_downloader]

while True:
    for coroutine in coroutines.copy():
        try:
            coroutine.send(None)
        except StopIteration:
            coroutines.remove(coroutine)
    if len(coroutines) == 0:
        break

Скачиваю часть 0
Скачиваю часть 0
Скачиваю часть 1
Скачиваю часть 1
Скачиваю часть 2
Скачиваю часть 2


In [ ]:
import time
import random
import asyncio

async def waiter(name):
    for _ in range(4):
        time_to_sleep = random.randint(1, 3) / 4
        time.sleep(time_to_sleep)
        # await asyncio.sleep(time_to_sleep)
        print(
            "{} waited {} seconds"
            "".format(name, time_to_sleep)
        )
   
async def main():
    await asyncio.wait([waiter("foo"), waiter("bar")])
   
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main())
    loop.close()

#### Future

Футуры (futures) — объекты, в которых хранится текущий результат выполнения какой-либо задачи. Это может быть информация о том, что задача ещё не обработана или уже полученный результат; а может быть вообще исключение.

Одна из особенностей футур, что мы можем запустить задачу на исполнение в одной корутине, а получить результат выполнения в другой. У футур есть 4 возможных состояния: + ожидание (pending) + выполнение (running) + выполнено (done) + отменено (cancelled)

In [30]:
import asyncio

async def slow_operation(future):
    await asyncio.sleep(1)
    future.set_result("Future is done!")

loop = asyncio.get_event_loop()
future = asyncio.Future()
asyncio.ensure_future(slow_operation(future))

loop.run_until_complete(future)
print(future.result())
loop.close()

RuntimeError: This event loop is already running

In [ ]:
import asyncio

async def set_after(delay, value):
    # Sleep for *delay* seconds.
    print('...')
    await asyncio.sleep(delay)

    # Set *value* as a result of *fut* Future.
    return value

async def main():
    # Run "set_after()" coroutine in a parallel Task.
    fut = asyncio.ensure_future(
        set_after(1, '... world'))

    print('hello ...')

    # Wait until *fut* has a result (1 second)
    await asyncio.wait_for(fut, None)
    # await fut
    
    # and print it.
    print(fut.result())

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
loop.close()

### Синхронные функции в asyncio

Event loop постоянно переключает контекст между всеми нашими корутинами и исполняет их последовательно — пока одна корутина ожидает ввода-вывода, вторую корутину event loop благополучно исполняет. Если код, который будет исполняться в корутине, будет блокирующим, то event loop не сможет делать переключения контекста. Для решения этой проблемы в asyncio существует метод run_in_executor. Он означает запустить код буквально в пуле потоков, который внутри
этого event loop-а автоматически будет создан (можно использовать собственный пул потоков или уже готовый по умолчанию).

In [ ]:
import asyncio

async def set_after(delay, value):
    # Sleep for *delay* seconds.
    await asyncio.sleep(delay)

    # Set *value* as a result of *fut* Future.
    return value

async def main():
    # Run "set_after()" coroutine in a parallel Task.
    fut = asyncio.ensure_future(
        set_after(1, '... world'))

    print('hello ...')

    # Wait until *fut* has a result (1 second)
    await asyncio.wait_for(fut, None)
    # and print it.
    print(fut.result())

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
loop.close()